# Imports

In [ ]:
import numpy as np 
import pandas as pd 
import os
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import optuna
from category_encoders import OneHotEncoder, MEstimateEncoder, CatBoostEncoder, OrdinalEncoder
from sklearn import set_config
import category_encoders
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, cross_val_predict
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import roc_auc_score, roc_curve, make_scorer, f1_score, accuracy_score, log_loss
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer, KNNImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import FunctionTransformer,StandardScaler, MinMaxScaler, LabelEncoder,PolynomialFeatures, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import auc, roc_auc_score
from sklearn.inspection import PartialDependenceDisplay
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.gaussian_process import GaussianProcessClassifier
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from colorama import Style, Fore
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool, cv
import warnings

warnings.filterwarnings("ignore", "use_inf_as_na")

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
sns.set_theme(style = 'white', palette = 'viridis')
pal = sns.color_palette('viridis')

pd.set_option('display.max_rows', 150)

# Loading Data

In [ ]:
train = pd.read_csv(r'/kaggle/input/playground-series-s4e2/train.csv',index_col='id')
test = pd.read_csv(r'/kaggle/input/playground-series-s4e2/test.csv', index_col='id')
sub  = pd.read_csv(r'/kaggle/input/playground-series-s4e2/sample_submission.csv')

In [ ]:
train.head(3)

In [ ]:
test.head()

## Descriptive Statistic


## train

In [ ]:
def summary(wdf) -> pd.DataFrame():
    df = wdf.copy()
    desc = pd.DataFrame(index = list(df))
    desc['type'] = df.dtypes
    desc['count'] = df.count()
    desc['nunique'] = df.nunique()
    desc['%unique'] = desc['nunique'] /len(df) * 100
    desc['null'] = df.isnull().sum()
    desc['%null'] = desc['null'] / len(df) * 100
    desc = pd.concat([desc,df.describe().T.drop('count',axis=1)],axis=1)
    desc = desc.sort_values(by=['type','null'])
    
    return desc

In [ ]:
desc_train = summary(train)
desc_train.style.background_gradient()

## test

In [ ]:
desc_test = summary(test)
desc_test.style.background_gradient()

* We have no null data
* low cardinality categorical attributes
* 7 possible classes

In [ ]:
TARGET = 'NObeyesdad'
NUM_COLS = train.select_dtypes('number').columns.tolist()
CAT_COLS = [c for c in train.columns if c not in NUM_COLS and c != TARGET]

# EDA


## Distribution of numerical variables

In [ ]:
import warnings
warnings.simplefilter('ignore',category=FutureWarning)
df = pd.concat([train[NUM_COLS].assign(Source='Train'),
                test[NUM_COLS].assign(Source='Test')],ignore_index=True)
fig, axs = plt.subplots(len(NUM_COLS),3, figsize=(16,len(NUM_COLS)*4),
                       gridspec_kw={'width_ratios': [0.8,0.2,0.2]}
                       )
for i, col in enumerate(NUM_COLS):
    ax = axs[i,0]
    sns.kdeplot(data=df[[col,'Source']],x=col,hue='Source',warn_singular=False,ax=ax)
    ax.set_title(f"\n{col}",fontsize = 9)
    ax.grid(visible=True, which = 'both', linestyle = '--', color='lightgrey', linewidth = 0.75)
    ax.set(xlabel = '', ylabel = '')
    
    ax = axs[i,1]
    sns.boxplot(data=df.loc[df.Source=='Train'],y=col,ax=ax,width=0.25, linewidth=0.90, fliersize=2.25)
    ax.set(xlabel = '', ylabel = '')
    ax.set_title("Train", fontsize = 9)
    
    ax = axs[i,2]
    sns.boxplot(data = df.loc[df.Source == 'Test', [col]], y = col, width = 0.25, linewidth = 0.90, fliersize= 2.25, color = '#ed7647', ax = ax)
    ax.set(xlabel = '', ylabel = '')
    ax.set_title("Test", fontsize = 9)

* training and testing have the same distribution
* Most ages are between 19 and 25 years old

## Target (NObeyesdad)

In [ ]:
ax = sns.countplot(x=train[TARGET])

total = len(train[TARGET])
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width() / 2., height + 0.1,
            '{:.2%}'.format(height / total),
            ha="center")
    ax.set_xticklabels(ax.get_xticklabels(),rotation=45)
plt.title('Percentage/number of classes',fontweight='bold')
plt.show()

* As seen previously, we have 7 classes, and they are not unbalanced, despite the Obesity_type_III class having a greater number.


## Categorical Features


In [ ]:
fig, ax = plt.subplots(8,1, figsize=(10,40), dpi=300)
ax = ax.flatten()
plt.subplots_adjust(wspace=0.3, hspace=0.2)
for i, col in enumerate(CAT_COLS):
    sns.countplot(x=train[col],hue=train[TARGET],ax=ax[i])
    ax[i].set_title(f'{col}')
    ax[i].set_xlabel(None)
    ax[i].set_xticklabels(ax[i].get_xticklabels(), rotation=90, ha='right',fontsize='small')
for j in range(len(CAT_COLS),len(ax)):
    ax[j].axis('off')
plt.tight_layout(h_pad=0.1)
plt.show()

*  The most used means of transport for all classes is public transport. Bicycles and motorcycles, the bar doesn't even appear on the graph.
* walking is always healthier.

## Age vs Target

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x=train['Age'],y=train[TARGET],showmeans=True,linewidth=0.5)
plt.show()

* Many outliers

# Cross Validation

In [ ]:
SEED = 42
SUBMIT = False
skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
oof, test_preds = {}, {}
le = LabelEncoder()
train[TARGET] = le.fit_transform(train[TARGET])

In [ ]:
def cross_validation(model, label):
    
    X= train.copy()
    y = train[TARGET]
    X = X.drop(TARGET,axis=1)
    scores = []
    oof_preds = np.zeros((len(X),7))
    for fold, (train_idx,val_idx) in enumerate(skf.split(X,y)):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        
        X_val   = X.iloc[val_idx]
        y_val   = y.iloc[val_idx]
                        
        model.fit(X_train,y_train)
        y_pred = model.predict_proba(X_val)        
        score = accuracy_score(y_val, np.argmax(y_pred,axis=1))    
        #score = log_loss(y_val, y_pred)
        scores.append(score)
        oof_preds[val_idx] = y_pred
        print(f"# Fold {fold}: Accuracy={score:.5f}")
    
    
    accuracy = accuracy_score(train[TARGET],np.argmax(oof_preds,axis=1))
    logloss = log_loss(train[TARGET],oof_preds)
    print(f"{Fore.GREEN}# Overall: accuracy={accuracy:.5f}"
      f" logloss={logloss:.5f} {label}")
    oof[label] = oof_preds
    
    if SUBMIT:
        X_tr = X
        y_tr = y
        
        model.fit(X_tr,y_tr)
        y_pred = model.predict_proba(X_val)
        test_preds[label] = y_pred
    
    

# Models

In [ ]:
for df in [train,test]:
    df[CAT_COLS] = df[CAT_COLS].astype('category')

In [ ]:
cross_validation(make_pipeline(OneHotEncoder(CAT_COLS),RandomForestClassifier(random_state=SEED)),'RF')

In [ ]:
cross_validation(CatBoostClassifier(cat_features=CAT_COLS,
                               iterations=200,
                               random_state=SEED,
                               silent=True,
                               loss_function='MultiClass'),'catboost')

In [ ]:
cross_validation(XGBClassifier(random_state=SEED,                               
                               objective='multi:softprob',
                               enable_categorical=True),'xgb')

# Ensemble

## Harding Voting

In [ ]:
Xens = np.column_stack([np.argmax(oof[label],axis=1) for label in oof.keys() if 'Ensemble' not in label])
print(f'hard voting {accuracy_score(train[TARGET],scipy.stats.mode(Xens,axis=1)[0])}')

oof['Ensemble'] = scipy.stats.mode(Xens,axis=1)[0]

# Stacking with Ridge

In [ ]:
Xs = np.hstack([oof[label] for label in oof.keys() if 'Ensemble' not in label])
model = Ridge()
oof['Ensemble (ridge)'] = cross_val_predict(model,
                                            Xs,label_binarize(train[TARGET], classes=[0, 1, 2, 3, 4, 5, 6]),
                                            cv=skf.split(Xs,train[TARGET]),method='predict')
oof['Ensemble (ridge)'] = oof['Ensemble (ridge)'].clip(0,1)

In [ ]:
result_list = []
for label in oof.keys():    
    if label!='Ensemble':
        score = accuracy_score(train[TARGET],np.argmax(oof[label],axis=1))
    else:
        score = accuracy_score(train[TARGET],oof[label])
    result_list.append((label,score))
result_df = pd.DataFrame(result_list,columns=['label','score'])
plt.figure(figsize=(18, len(result_df) * 0.4 + 0.4))
bars = plt.barh(np.arange(len(result_df)), result_df.score, color='lightgreen')
plt.gca().bar_label(bars, fmt='%.5f')
plt.yticks(np.arange(len(result_df)), result_df.label)
plt.gca().invert_yaxis()
plt.xlabel(f'log loss (low is better)')
plt.show()

# Calibration

In [ ]:
plt.figure(figsize=(15,16))
for target in range(7):    
    plt.subplot(3,3,target+1)
    CalibrationDisplay.from_predictions(train[TARGET]==target,
                                        oof['Ensemble (ridge)'][:,target],
                                        n_bins=50,
                                        strategy='quantile',
                                        name='Ensemble (ridge)',
                                        ax=plt.gca()
                                       )
    plt.xlabel(f'Mean predicted probability for {le.classes_[target]}')
    plt.ylabel(f'Fraction of positives for {le.classes_[target]}')
plt.suptitle('Calibration')
plt.show()